In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers datasets accelerate evaluate jiwer
!pip install -q soundfile librosa openpyxl requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 65.1 MB/s eta 0:00:00


In [3]:
# CELL 1
!pip install -q transformers datasets evaluate jiwer pydub accelerate torchaudio

In [ ]:
import torch
from datasets import load_dataset
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import evaluate

# 1. Load the FLEURS test dataset for Hindi
# 1. Load the FLEURS test dataset for Hindi
# Load the Parquet-converted branch directly, bypassing the broken script
fleurs_test = load_dataset(
    "google/fleurs",
    data_dir="hi_in", # Note: if this throws a config error, change "hi_in" to data_dir="hi_in"
    split="test",
    revision="refs/convert/parquet"
)

# 2. Load the pretrained Whisper-small model and processor
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Hindi", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to("cuda")

wer_metric = evaluate.load("wer")

def evaluate_baseline(batch):
    audio = batch["audio"]
    # Extract log-Mel spectrogram features
    input_features = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt").input_features.to("cuda")

    with torch.no_grad():
        predicted_ids = model.generate(input_features)

    # Decode the predicted ids to text
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    batch["prediction"] = transcription
    batch["reference"] = batch["raw_transcription"]
    return batch

print("Evaluating baseline on FLEURS test set...")
# Run inference over the dataset
results = fleurs_test.map(evaluate_baseline)

# Calculate and report the baseline WER
baseline_wer = wer_metric.compute(predictions=results["prediction"], references=results["reference"])
print(f"Pretrained Whisper-small Baseline WER: {baseline_wer * 100:.2f}%")

hi_in/train/0000.parquet:   0%|          | 0.00/502M [00:00<?, ?B/s]

hi_in/train/0001.parquet:   0%|          | 0.00/513M [00:00<?, ?B/s]

hi_in/train/0002.parquet:   0%|          | 0.00/501M [00:00<?, ?B/s]

hi_in/train/0003.parquet:   0%|          | 0.00/14.0M [00:00<?, ?B/s]

hi_in/validation/0000.parquet:   0%|          | 0.00/162M [00:00<?, ?B/s]

...

generation_config.json: 0.00B [00:00, ?B/s]

Evaluating baseline on FLEURS test set...


Map:   0%|          | 0/418 [00:00<?, ? examples/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log

Pretrained Whisper-small Baseline WER: 86.99%


In [ ]:
# CELL 2 (FINAL JSON FIX)
import pandas as pd
import requests
import json
import os
from pydub import AudioSegment
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor
import shutil

# 1. Setup Folders
INPUT_FILE = "/content/FT Data.xlsx"
OUTPUT_DIR = "/content/whisper_training_data"
os.makedirs(f"{OUTPUT_DIR}/audio", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/temp", exist_ok=True)

# 2. Function to fix the broken Josh Talks URLs
def fix_urls(row):
    original_url = str(row['transcription_url_gcp'])
    try:
        folder_id = original_url.split('/')[-2]
    except IndexError:
        folder_id = "UNKNOWN"

    rec_id = str(row['recording_id'])
    base_url = f"https://storage.googleapis.com/upload_goai/{folder_id}/"

    row['fixed_rec_url'] = f"{base_url}{rec_id}_audio.wav"
    row['fixed_json_url'] = f"{base_url}{rec_id}_transcription.json"
    return row

# 3. Function to download files
def download_file(url, local_filename):
    try:
        with requests.get(url, stream=True, timeout=10) as r:
            r.raise_for_status()
            with open(local_filename, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        return True
    except:
        return False

# 4. The Audio Splicer Engine
def process_recording(row):
    rec_id = str(row['recording_id'])
    local_wav = f"{OUTPUT_DIR}/temp/{rec_id}.wav"
    local_json = f"{OUTPUT_DIR}/temp/{rec_id}.json"
    chunks_meta = []

    if not (download_file(row['fixed_rec_url'], local_wav) and download_file(row['fixed_json_url'], local_json)):
        if os.path.exists(local_wav): os.remove(local_wav)
        if os.path.exists(local_json): os.remove(local_json)
        return []

    try:
        audio = AudioSegment.from_wav(local_wav)
        with open(local_json, 'r', encoding='utf-8') as f:
            utterances = json.load(f) # <--- FIX: Directly load the list

        if not isinstance(utterances, list) or not utterances:
            return []

        current_chunk_text = []
        # <--- FIX: Use 'start' instead of 'start_time'
        current_chunk_start_ms = float(utterances[0].get('start', 0)) * 1000
        chunk_index = 0

        for i, utt in enumerate(utterances):
            # <--- FIX: Use 'start', 'end', and 'text' keys
            start_ms = float(utt.get('start', 0)) * 1000
            end_ms = float(utt.get('end', 0)) * 1000
            text = str(utt.get('text', ''))

            proposed_duration_ms = end_ms - current_chunk_start_ms

            if (proposed_duration_ms <= 29500):
                current_chunk_text.append(text)
            else:
                chunk_filename = f"{rec_id}_chunk_{chunk_index:03d}.wav"
                chunk_path = f"{OUTPUT_DIR}/audio/{chunk_filename}"

                previous_end_ms = float(utterances[i-1].get('end', 0)) * 1000
                current_chunk_audio = audio[current_chunk_start_ms : previous_end_ms]
                current_chunk_audio.set_frame_rate(16000).export(chunk_path, format="wav")

                chunks_meta.append({
                    'audio': chunk_path,
                    'sentence': " ".join(current_chunk_text)
                })

                chunk_index += 1
                current_chunk_text = [text]
                current_chunk_start_ms = start_ms

        if current_chunk_text:
            chunk_filename = f"{rec_id}_chunk_{chunk_index:03d}.wav"
            chunk_path = f"{OUTPUT_DIR}/audio/{chunk_filename}"
            final_end_ms = float(utterances[-1].get('end', 0)) * 1000
            final_audio = audio[current_chunk_start_ms : final_end_ms]
            final_audio.set_frame_rate(16000).export(chunk_path, format="wav")

            chunks_meta.append({
                'audio': chunk_path,
                'sentence': " ".join(current_chunk_text)
            })

    except Exception as e:
        pass
    finally:
        if os.path.exists(local_wav): os.remove(local_wav)
        if os.path.exists(local_json): os.remove(local_json)

    return chunks_meta

# 5. Execute Pipeline
print("Loading Excel file and fixing URLs...")
df = pd.read_excel(INPUT_FILE)
df = df.apply(fix_urls, axis=1)

print(f"Downloading and splicing {len(df)} audio files... This will take a few minutes.")
all_chunks_metadata = []

with ThreadPoolExecutor(max_workers=10) as executor:
    results = list(tqdm(executor.map(process_recording, [row for _, row in df.iterrows()]), total=len(df)))

for result_list in results:
    if result_list:
        all_chunks_metadata.extend(result_list)

new_df = pd.DataFrame(all_chunks_metadata)
clean_csv_path = f"{OUTPUT_DIR}/clean_whisper_data.csv"
new_df.to_csv(clean_csv_path, index=False)
shutil.rmtree(f"{OUTPUT_DIR}/temp")

print(f"Done! Created {len(new_df)} perfectly sized training chunks.")

Loading Excel file and fixing URLs...


  0%|          | 0/104 [00:00<?, ?it/s]

Done! Created 2522 perfectly sized training chunks.


## Step 3 — Preprocess the data (Q1a)

This is what they want you to document. Key steps:

Resample all audio to 16kHz (Whisper requirement)

Filter out recordings that are too short (<1s) or too long (>30s)

Normalize transcription text (strip extra spaces, consistent Unicode normalization for Devanagari)

Convert audio to log-mel spectrograms using Whisper's feature extractor
Create a Hugging Face Dataset object

In [1]:
# CELL 3
import pandas as pd
import unicodedata
import re
from datasets import Dataset, Audio

# 1. Load the cleanly sliced CSV
df = pd.read_csv("/content/whisper_training_data/clean_whisper_data.csv")

# 2. Normalize Devanagari Text
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    # NFC normalization ensures consistent rendering of Hindi characters + matras
    text = unicodedata.normalize("NFC", text)
    # Strip leading/trailing spaces and remove multiple internal spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Normalizing Devanagari text...")
df['sentence'] = df['sentence'].apply(normalize_text)

# Drop any rows that ended up completely empty after cleaning
df = df[df['sentence'] != ""]

# 3. Create Hugging Face Dataset & Resample
hf_dataset = Dataset.from_pandas(df)

# This line automatically resamples all audio to 16kHz on the fly when loaded
hf_dataset = hf_dataset.cast_column("audio", Audio(sampling_rate=16000))

print(f"Loaded {len(hf_dataset)} chunks into the Hugging Face Dataset.")

Normalizing Devanagari text...
Loaded 2522 chunks into the Hugging Face Dataset.


In [2]:
# CELL 4
from transformers import WhisperForConditionalGeneration, WhisperProcessor


print("Loading Whisper Processor...")
# Set language to Hindi and task to transcribe
#processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Hindi", task="transcribe")
# 2. Load the pretrained Whisper-small model and processor
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Hindi", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to("cuda")


print("model loaded")

Loading Whisper Processor...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

model loaded


## saving the dataset into drive

In [ ]:
# 2. Define exactly what we are saving and where it's going
SOURCE_DIR = "/content/whisper_training_data"
# This creates a specific folder in your Drive so it doesn't clutter your main directory
DESTINATION_DIR = "/content/drive/MyDrive/Josh_Talks_Whisper_Data"

print(f"\nBacking up data to: {DESTINATION_DIR}")
print("This might take a couple of minutes depending on the number of chunks...")

# 3. Copy the entire folder and its contents
try:
    shutil.copytree(SOURCE_DIR, DESTINATION_DIR, dirs_exist_ok=True)

    # Quick verification
    saved_files = os.listdir(f"{DESTINATION_DIR}/audio")
    print(f"\n✅ SUCCESS: Successfully backed up {len(saved_files)} audio chunks and the CSV map to your Google Drive!")
except Exception as e:
    print(f"\n❌ Error during backup: {e}")


Backing up data to: /content/drive/MyDrive/Josh_Talks_Whisper_Data
This might take a couple of minutes depending on the number of chunks...

✅ SUCCESS: Successfully backed up 2522 audio chunks and the CSV map to your Google Drive!


In [8]:
# RESTORE DATA FROM GOOGLE DRIVE
from google.colab import drive
import shutil
import os

# 1. Mount Drive (if not already mounted)
drive.mount('/content/drive')

SOURCE_DIR = "/content/drive/MyDrive/Josh_Talks_Whisper_Data"
DESTINATION_DIR = "/content/whisper_training_data"

print("Restoring audio chunks from Google Drive backup...")

try:
    # Copy everything back to the local Colab disk
    shutil.copytree(SOURCE_DIR, DESTINATION_DIR, dirs_exist_ok=True)

    # Verify it worked
    restored_files = os.listdir(f"{DESTINATION_DIR}/audio")
    print(f"✅ SUCCESS: Restored {len(restored_files)} audio chunks back to Colab!")

except FileNotFoundError:
    print("❌ ERROR: Could not find the backup in Google Drive. You may need to re-run Cell 2 to generate the chunks again.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Restoring audio chunks from Google Drive backup...
✅ SUCCESS: Restored 2522 audio chunks back to Colab!


In [5]:
# CELL 5 (ULTIMATE RAM-SAFE VERSION)
import warnings
import gc
# Ignore librosa/torchaudio warnings during mapping
warnings.filterwarnings("ignore")

# 1. Clean house before we start
gc.collect()

# 2. Strict Duration Filtering
def filter_by_length(batch):
    audio = batch["audio"]
    # Calculate duration: total samples / sampling rate (16000)
    duration = len(audio["array"]) / audio["sampling_rate"]
    # Keep only audio strictly between 1.0 and 30.0 seconds
    return 1.0 <= duration <= 30.0

print("Filtering out audio chunks < 1s or > 30s...")
filtered_dataset = hf_dataset.filter(filter_by_length)
print(f"Chunks remaining after length filtering: {len(filtered_dataset)}")

# 3. Feature Extraction (Log-Mel Spectrograms & Tokenization)
def prepare_dataset(batch):
    audio = batch["audio"]

    # Process audio into log-mel spectrogram features
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    # Process text into token IDs
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

print("Extracting Log-Mel features and tokenizing text...")
print("Writing chunks directly to disk to protect RAM. This will take a few minutes!")

# ---> THE RAM FIX IS HERE <---
final_dataset = filtered_dataset.map(
    prepare_dataset,
    remove_columns=filtered_dataset.column_names,
    keep_in_memory=False,  # Forces HF to write to disk instead of hoarding RAM
    writer_batch_size=50   # Dumps chunks to disk every 50 files to keep RAM empty
)

# 4. NO SPLITTING
# As per Josh Talks instructions, we keep 100% of this data for fine-tuning.
# We will load FLEURS separately for the evaluation step.

print("\n✅ Final Training Dataset Ready for GPU Fine-Tuning!")
print(final_dataset)

Filtering out audio chunks < 1s or > 30s...


Filter:   0%|          | 0/2522 [00:00<?, ? examples/s]

Chunks remaining after length filtering: 2506
Extracting Log-Mel features and tokenizing text...
Writing chunks directly to disk to protect RAM. This will take a few minutes!


Map:   0%|          | 0/2506 [00:00<?, ? examples/s]


✅ Final Training Dataset Ready for GPU Fine-Tuning!
Dataset({
    features: ['input_features', 'labels'],
    num_rows: 2506
})


## Data Collator, Metrics, and FLEURS Preparation


To train effectively, we need a custom "Data Collator" that dynamically pads the audio sequences and text sequences (since they are different lengths). We also need to download the FLEURS Hindi test set and process it using the exact same log-mel extraction we used for our training data.

In [6]:
# CELL 6 (ULTIMATE RAM SAVER FOR FLEURS)
import torch
import gc
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate
from datasets import load_dataset, Audio

# 1. Clean house: Free up any lingering junk memory before we start
gc.collect()
torch.cuda.empty_cache()

# 2. Define the Data Collator
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# 3. Define the WER Metric
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

# 4. Prepare the FLEURS Test Set
print("Downloading and preparing the FLEURS Test Set for evaluation...")

fleurs_test = load_dataset(
    "google/fleurs",
    data_dir="hi_in",
    split="test",
    revision="refs/convert/parquet"
)
fleurs_test = fleurs_test.cast_column("audio", Audio(sampling_rate=16000))

def prepare_fleurs(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = processor.tokenizer(batch["raw_transcription"]).input_ids
    return batch

print("Extracting features for FLEURS (Writing to disk to save RAM)...")

# ---> THE RAM FIX IS HERE <---
fleurs_test_prepared = fleurs_test.map(
    prepare_fleurs,
    remove_columns=fleurs_test.column_names,
    keep_in_memory=False,   # Forces HF to write to disk instead of hoarding RAM
    writer_batch_size=25    # Dumps chunks to disk every 25 files to keep RAM empty
)
print("✅ FLEURS Test Set Prepared!")

Extracting features for FLEURS (Writing to disk to save RAM)...
✅ FLEURS Test Set Prepared!


In [7]:
# Check the exact number of chunks in final_dataset
print(f"Total chunks in final_dataset: {len(final_dataset)}")

Total chunks in final_dataset: 2506


In [8]:
# CHECK YOUR TRAINING DATA CELL
import pandas as pd
import random
import IPython.display as ipd
import os

# 1. Load your master map
csv_path = "/content/whisper_training_data/clean_whisper_data.csv"
df = pd.read_csv(csv_path)

# 2. Pick a random row
random_index = random.randint(0, len(df) - 1)
sample = df.iloc[random_index]

audio_path = sample['audio']
text = sample['sentence']

print("🔍 INSPECTING RANDOM TRAINING CHUNK")
print("-" * 50)
print(f"📂 File: {os.path.basename(audio_path)}")
print(f"📝 Text: {text}")
print("-" * 50)

# 3. Create a playable audio widget in Colab
if os.path.exists(audio_path):
    ipd.display(ipd.Audio(audio_path))
else:
    print("❌ Audio file not found! (Did your Colab wipe the files again?)")

🔍 INSPECTING RANDOM TRAINING CHUNK
--------------------------------------------------
📂 File: 989901_chunk_028.wav
📝 Text: जस्ट लाइक की आपके जैसा मुझे भी कोई काफी करुणा है मैं दिल में भी काफी करुणा के भाव उत्पन्न होते है सामने वाले की हेल्प करना मुझे अच्छा लगता हैं बाद में भले मुझे महसूस करा देते है लोग की नहीं करना चाहिए था इसका हेल्प REDACTED हां हां बिल्कुल
--------------------------------------------------


In [8]:
# CELL 7 (THE NUCLEAR TOKEN FIX)
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

MAX_TOKENS = 440

# 1. The Nuclear Fix: Forcefully truncate ALL labels in both datasets
def truncate_labels(batch):
    # If a label is 469 tokens, this physically cuts it down to exactly 440.
    batch["labels"] = batch["labels"][:MAX_TOKENS]
    return batch

print("Enforcing strict token truncation on Training Data...")
bulletproof_train_dataset = final_dataset.map(truncate_labels)

print("Enforcing strict token truncation on FLEURS Test Data...")
bulletproof_eval_dataset = fleurs_test_prepared.map(truncate_labels)

# 2. Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-josh-talks-hindi",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_steps=100,
    num_train_epochs=2,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="epoch",
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    generation_max_length=225,
    save_strategy="epoch",
    logging_steps=50,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

# 3. Initialize Trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=bulletproof_train_dataset, # Using the physically truncated data
    eval_dataset=bulletproof_eval_dataset,   # Using the physically truncated FLEURS data
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

# 4. Start Fine-Tuning
print("\n🚀 Starting Fine-Tuning! Token limits are now physically enforced.")
trainer.train()

# 5. Calculate Final Fine-Tuned WER
print("🎙️ Calculating Final Fine-Tuned WER...")
final_results = trainer.evaluate()
print(f"📊 FINE-TUNED WER: {final_results['eval_wer']:.2f}%")

Enforcing strict token truncation on Training Data...


Map:   0%|          | 0/2506 [00:00<?, ? examples/s]

Enforcing strict token truncation on FLEURS Test Data...


Map:   0%|          | 0/418 [00:00<?, ? examples/s]


🚀 Starting Fine-Tuning! Token limits are now physically enforced.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


🎙️ Calculating Final Fine-Tuned WER...


📊 FINE-TUNED WER: 46.74%


In [9]:
import os

# 2. Define where you want it saved in your Drive
drive_save_path = "/content/drive/MyDrive/whisper-josh-talks-hindi-finetuned"
os.makedirs(drive_save_path, exist_ok=True)

print(f"Saving model and processor to: {drive_save_path} ...")

# 3. Save the Model Weights (The Brain)
trainer.save_model(drive_save_path)

# 4. Save the Processor (The Ears and Mouth)
processor.save_pretrained(drive_save_path)

print("✅ SUCCESS! Your fine-tuned model is permanently saved to Google Drive.")

Saving model and processor to: /content/drive/MyDrive/whisper-josh-talks-hindi-finetuned ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ SUCCESS! Your fine-tuned model is permanently saved to Google Drive.


## Part D: Systematic Sampling Strategy &

Extraction CodeTo avoid cherry-picking, your strategy will be Systematic Sampling by $N$-th Interval.We will run your fine-tuned model over a chunk of the FLEURS test set.

We will calculate the WER for each individual sentence.

We will filter out the perfect predictions (WER = 0%).

From the remaining flawed predictions, we will
sort them by WER and select every $N$-th item until we have exactly 25 diverse examples, ranging from minor typos to massive hallucinations.

In [11]:
# SYSTEMATIC ERROR SAMPLING
import pandas as pd
import evaluate
from tqdm import tqdm

# 1. Load the WER metric
metric = evaluate.load("wer")

print("Generating predictions on a subset of FLEURS to find errors...")
results = []

# Evaluate the first 150 samples of the FLEURS test set
# (We don't need the whole thing just to find 25 errors)
for i in tqdm(range(150)):
    sample = fleurs_test_prepared[i]
    reference_text = processor.tokenizer.decode(sample["labels"], skip_special_tokens=True)

    # Generate prediction using your fine-tuned model
    input_features = torch.tensor(sample["input_features"]).unsqueeze(0).to("cuda")
    predicted_ids = model.generate(input_features, max_length=225)
    predicted_text = processor.tokenizer.decode(predicted_ids[0], skip_special_tokens=True)

    # Calculate WER for this specific utterance
    try:
        error_rate = metric.compute(predictions=[predicted_text], references=[reference_text])
    except:
        error_rate = 1.0 # Handle edge cases where prediction is completely empty

    results.append({
        "Reference": reference_text,
        "Prediction": predicted_text,
        "WER": error_rate
    })

# 2. Filter and Sample
df_results = pd.DataFrame(results)

# Keep only the ones where the model made a mistake (WER > 0)
df_errors = df_results[df_results["WER"] > 0.0].sort_values(by="WER").reset_index(drop=True)

# Systematic Sampling: Pick 25 items evenly spaced across the error distribution
step = max(1, len(df_errors) // 25)
sampled_errors = df_errors.iloc[::step].head(25).reset_index(drop=True)

# 3. Export to CSV for your assignment documentation
sampled_errors.to_csv("systematic_25_errors.csv", index=False)

print(f"\n✅ Successfully sampled {len(sampled_errors)} errors systematically!")
print("Here are the first 3 examples to look at:")
print(sampled_errors.head(3))

Generating predictions on a subset of FLEURS to find errors...


100%|██████████| 150/150 [07:53<00:00,  3.16s/it]


✅ Successfully sampled 25 errors systematically!
Here are the first 3 examples to look at:
                                           Reference  \
0  विखंडन बम इस सिद्धांत पर काम करता है कि कई प्र...   
1  हर कोई समाज से जुड़ा होता है और ट्रांसपोर्ट सि...   
2  यहूदी और गैर-यहूदी की तरह, अभी भी ऐसे कई पुरुष...   

                                          Prediction      WER  
0  बिखंडन बम इस सिधान्त पर काम करता है कि कई प्रो...  0.16000  
1  फर कोई समाज से जुड़ा होता है और ट्रांस्पूर्ट स...  0.24000  
2  येहूदी और गैयाव यहूदी कितारा अभी भी ऐसे कई पूर...  0.27027  


# link to the answers of question 1:

https://docs.google.com/document/d/1vmRjqcIEQ1Pnj06LzQx-7CRJS-stgmyG1sVOgcXKu24/edit?usp=sharing

 # g) Implement at least one of your proposed fixes within the assignment timeframe. Show before/after results on a targeted subset of your error examples.

In [12]:
# PART G: IMPLEMENTING THE TEXT NORMALIZER
import pandas as pd
import re
import evaluate

# 1. Load the Word Error Rate metric
metric = evaluate.load("wer")

# 2. Load your 25 sampled errors
df_errors = pd.read_csv("systematic_25_errors.csv")

# 3. Define the strict Hindi Text Normalizer
def normalize_hindi_text(text):
    if not isinstance(text, str):
        return ""

    # Strip out all English and Hindi punctuation marks
    # (commas, periods, question marks, quotes, brackets, and the Hindi full stop '।')
    text = re.sub(r'[।॥!,\.\?\'"\[\]\(\)\{\}:;\-]', ' ', text)

    # Remove hidden formatting characters (Zero-width joiners used in typing Hindi)
    text = re.sub(r'[\u200b\u200c\u200d\uFEFF]', '', text)

    # Collapse multiple spaces into a single space and trim edges
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# 4. Apply the normalizer to both columns
df_errors['Norm_Reference'] = df_errors['Reference'].apply(normalize_hindi_text)
df_errors['Norm_Prediction'] = df_errors['Prediction'].apply(normalize_hindi_text)

# 5. Calculate the "After" WER
def calculate_new_wer(row):
    try:
        # If reference is empty after stripping, ignore it to prevent division by zero
        if len(row['Norm_Reference'].strip()) == 0:
            return row['WER']
        return metric.compute(predictions=[row['Norm_Prediction']], references=[row['Norm_Reference']])
    except:
        return row['WER']

df_errors['New_WER'] = df_errors.apply(calculate_new_wer, axis=1)

# 6. Show the Before and After Results
print("🎯 TEXT NORMALIZATION RESULTS (BEFORE vs AFTER)")
print("-" * 60)

# Calculate the average drop across the 25 samples
avg_before = df_errors['WER'].mean() * 100
avg_after = df_errors['New_WER'].mean() * 100

print(f"Average WER (Before): {avg_before:.2f}%")
print(f"Average WER (After):  {avg_after:.2f}%\n")
print(f"📉 Absolute Improvement: {(avg_before - avg_after):.2f}%")
print("-" * 60)

# Let's look at 3 specific examples where punctuation was inflating the error
examples_to_show = df_errors.sample(3)

for index, row in examples_to_show.iterrows():
    print(f"Sample #{index}")
    print(f"Original Ref: {row['Reference']}")
    print(f"Original Prd: {row['Prediction']}")
    print(f"   -> WER: {row['WER']:.2f}")
    print(f"Norm'd Ref:   {row['Norm_Reference']}")
    print(f"Norm'd Prd:   {row['Norm_Prediction']}")
    print(f"   -> NEW WER: {row['New_WER']:.2f}\n")

# Save this out so you can put it in your final assignment report!
df_errors.to_csv("normalized_25_errors_results.csv", index=False)

🎯 TEXT NORMALIZATION RESULTS (BEFORE vs AFTER)
------------------------------------------------------------
Average WER (Before): 45.45%
Average WER (After):  38.36%

📉 Absolute Improvement: 7.09%
------------------------------------------------------------
Sample #14
Original Ref: जिस समय घटनाएँ हुई उसके युग को सामान्य रूप से यूरोप के इतिहास में 11वीं, 12वीं और 13वीं सदी (AD 1000-1300) में यूरोपियन इतिहास के उच्च मध्य युग की अवधि के रूप में बताया जाता है.
Original Prd: जिस समय घटनाई हुई उसके योग को समाने रूप से यूरोप के इतिहास में यारवी बारवी और तेरवी सदी एडी थाउजन तू थर्टीन हंटरेड में यूरोपियान इतिहास के उच्छ मद्यवी की अबदी के रूप में बताया जाता है।
   -> WER: 0.47
Norm'd Ref:   जिस समय घटनाएँ हुई उसके युग को सामान्य रूप से यूरोप के इतिहास में 11वीं 12वीं और 13वीं सदी AD 1000 1300 में यूरोपियन इतिहास के उच्च मध्य युग की अवधि के रूप में बताया जाता है
Norm'd Prd:   जिस समय घटनाई हुई उसके योग को समाने रूप से यूरोप के इतिहास में यारवी बारवी और तेरवी सदी एडी थाउजन तू थर्टीन हंटरेड में यूर

Actually, look closely at the numbers! The WER did change. It dropped from 0.47 (47%) down to 0.43 (43%).

That is an 8.5% relative improvement! Why did it drop? Because the normalizer successfully deleted the commas , and the brackets ( ) that Whisper had no way of hearing.

however the biggest chunk of the error is still sitting right there. Look at this exact alignment:

Reference wants: 11वीं 12वीं और 13वीं सदी AD 1000 1300

Whisper heard: यारवी बारवी और तेरवी सदी एडी थाउजन तू थर्टीन हंटरेड

we will solve this in  next question as the next question fully depends on it